# The Knowledge Benchmarks: An Evolution Story
*A deep, benchmark-by-benchmark evolution story — MMLU (2020), TruthfulQA (2021), AGIEval (2023), GPQA (2023), MMLU-Pro (2024), Humanity's Last Exam (2025), and SimpleQA (2024) — each covered in full: history and lineage, dataset and task, scoring methodology, run configuration, what it doesn't measure, and known issues and criticisms, with every citation verified against primary sources*

## Why knowledge was the first capability anyone tested

The knowledge capability measures how much factual, real-world information a model retained from training — how much world knowledge sits encoded in its weights, sometimes called its **parametric knowledge**.

This is arguably the most fundamental capability an LLM can have. When models were first trained on massive scrapes of internet text, the most basic expectation was simple: a model trained on that much of the internet should be able to answer questions about more or less anything on it. Other capabilities that get discussed today — reasoning, coding — came later, and largely **emerged** as a side effect of scale rather than being explicitly targeted from the start. Reasoning showed up gradually as models got bigger, without anyone specifically engineering it in; coding ability emerged similarly, once enough code made it into training data. Knowledge, by contrast, was the original, most direct expectation placed on a freshly trained model — which is exactly why it's the capability with the longest, most developed history of dedicated benchmarks.

Rather than covering a handful of famous benchmarks from every capability in brief, this notebook follows one capability's benchmarks in full: how they evolved, what problem each new one solved, and why it's a genuinely useful way to build real understanding of how benchmarking as a field actually progresses. The same kind of evolution has played out for every other capability — this is simply the most complete one to walk through end to end.

## The starting point: testing without a system

In the earliest days of large language models — GPT-2 and GPT-3-era models, trained on massive internet-scale data — testing what a model actually knew was informal. People would ask a model random questions across different domains and see whether it answered correctly. It usually did reasonably well, which felt reassuring, but this approach couldn't actually quantify *how much* knowledge a model had retained, or allow one model to be compared against another in any rigorous way. What was missing was a proper, systematic evaluation process — which is exactly the gap MMLU was built to fill.

## 2020: MMLU — the mother of all knowledge benchmarks

**MMLU** (Massive Multitask Language Understanding) is a dataset of roughly 14,000 multiple-choice questions spanning 57 subjects — everything from elementary mathematics to law, medicine, and professional ethics. It measures **breadth** of knowledge specifically — not depth, not reasoning, just how wide a slice of human knowledge a model can correctly answer questions about. **Paper:** [arxiv.org/abs/2009.03300](https://arxiv.org/abs/2009.03300)

It's earned the title "mother of all benchmarks" for knowledge because nearly everything covered later in this notebook is, directly or indirectly, a response to something MMLU couldn't do. Its own history is worth walking through in detail before getting to what came after it, since it's a genuinely complete case study of how a benchmark is born, rises, and eventually gets retired.

### History and lineage

| Period | What happened |
|---|---|
| **September 2020** | MMLU is published. At launch, most existing models score close to the 25% random-chance floor; the best model at the time, GPT-3, reaches 43.9%. Human domain experts, tested on the same questions, average around 89.8%. |
| **2021–2022** | MMLU becomes the standard knowledge benchmark during the "scaling laws" era, when the dominant strategy across the field was simply to keep increasing parameter count — 175 billion, then 350 billion, then further — on the general expectation that capability would scale with size. Models from this period (Gopher, Chinchilla, PaLM, among others) all reported their MMLU scores as a headline metric. |
| **2023** | GPT-4 scores roughly 86% — closing most of the gap to the human expert baseline, and marking the point where MMLU's usefulness as a differentiator started to decline. |
| **2024** | Frontier models across every major lab converge into a tight 86–92% cluster, making it increasingly difficult to tell a strong model from an exceptional one using MMLU alone. Around this time, a detailed manual review of MMLU's questions found that roughly 6.5% contain errors — either an incorrect labeled answer or no correct answer among the options at all — meaning no model could ever score meaningfully above roughly 92–93%, regardless of actual capability. |
| **2025** | Frontier labs largely stop reporting MMLU as a primary benchmark, having shifted to newer benchmarks like GPQA and HLE that could still meaningfully separate models. |

Every benchmark's saturation story from earlier applies here concretely: MMLU didn't fail because it was a bad benchmark — it succeeded so completely, for so long, that the field eventually outgrew what it could measure.

### Dataset and task

MMLU's 57 subjects are grouped into four broad categories: **Humanities**, **Social Sciences**, **STEM**, and **Other**. Its questions were sourced from real exams — including the GRE, the USMLE (the US medical licensing exam), and AP exams — along with other credible sources like textbooks, rather than being written from scratch specifically for the benchmark.

A representative question, from the college physics subject:

> The muon decays with a characteristic lifetime of about 10⁻⁶ seconds into an electron...

Each question comes with four answer options and one labeled correct answer. The task given to a model is simple: read the question and its four options, and select the correct one — typically alongside five worked examples shown first (few-shot prompting) so the model sees the expected format before answering the actual question.

### Scoring methodology

MMLU can be scored two different ways, and they don't always agree:

- **Direct generation.** The model is asked to output a single letter — A, B, C, or D — and that letter is checked directly against the correct answer.
- **Log-likelihood comparison.** Rather than generating text, the model's internal probability for each of the four options is extracted directly, and whichever option it assigns the highest probability to is treated as its answer — without the model ever explicitly "choosing" anything in its output.

These two methods can disagree by 1–3 percentage points for the same model on the same questions — GPT-4, for instance, has been reported around 84% under direct generation and around 87% under log-likelihood comparison. Neither method is "more correct" than the other, but it means a reported MMLU score is only meaningfully comparable to another score if both used the same scoring method.

Results can also be reported two ways: **overall accuracy** across all 14,000 questions combined, or **per-subject accuracy** — a separate score for biology, physics, law, and so on — which reveals whether a model's overall strength is even across subjects or concentrated in a few.

### Run configuration

MMLU is unusually sensitive to exact prompt formatting — small wording changes in the system prompt can shift a model's score noticeably, which has occasionally been exploited to inflate reported numbers. The standard, broadly agreed-on configuration is:

- **5-shot prompting** — five worked examples shown before the actual question.
- **No chain-of-thought** — the model isn't asked to reason step by step, since MMLU is meant to test recall and direct knowledge application rather than reasoning depth. (When chain-of-thought is allowed anyway, scores tend to improve slightly, simply because the model spends more effort per answer.)
- **Temperature 0**, for consistent, reproducible outputs.
- **Pass@1** — one attempt per question, scored correct or incorrect.
- **No tools** — no web search, no code execution, since the point is testing what the model itself knows.

Comparing two models fairly on MMLU requires holding every one of these constant across both runs — the format-sensitivity issue means even small deviations can produce misleadingly different scores for models of genuinely similar capability.

### What MMLU doesn't measure

MMLU is deliberately narrow in scope, and it's worth being explicit about what falls outside it:

- **Reasoning depth.** MMLU isn't built to test multi-step reasoning — it's a knowledge-recall benchmark first.
- **Calibration.** It doesn't test whether a model actually knows when it doesn't know something — whether it can distinguish a confident correct answer from a confident guess. This is a truthfulness concern, which is exactly the gap TruthfulQA and SimpleQA exist to cover.
- **Open-ended generation.** Every question is multiple-choice with four fixed options — MMLU never tests what a model would say if it had to generate a free-form answer without anything to select from.
- **Multilingual or non-Western knowledge.** MMLU is English-only, exam-style, and its content skews toward Western curricula. It says little about a model's knowledge of, say, Chinese history or Indian civics specifically.

### Known issues and criticisms

- **Label errors.** As noted above, roughly 6.5% of MMLU's questions are flawed — either mislabeled or missing a genuinely correct option among the four choices — which caps the realistic maximum score below 100% regardless of model quality.
- **Contamination.** MMLU's dataset has been public since 2020, which means it's extremely likely to have been swept into the pretraining data of virtually every model trained since — the same contamination issue covered earlier, but especially severe here given how long the dataset has been circulating.
- **Prompt format sensitivity.** Because scores can shift meaningfully based on small formatting choices, this has occasionally been exploited — tuning prompt wording specifically to inflate a reported score, rather than reflecting genuine capability differences.

### How questions were selected

MMLU's inclusion bar, by the standards of later benchmarks, was fairly permissive: a valid question needed four answer options, a known correct answer, and a clear connection to a specific subject area — ideally sourced from a credible reference like an exam or textbook, or contributed by a subject expert. That was largely sufficient for inclusion.

This stands in sharp contrast to benchmarks built years later once contamination and quality concerns were better understood — Humanity's Last Exam, for instance, put contributed questions through multiple rounds of expert review and offered real prize money for questions accepted into the final set. MMLU's more permissive selection process is itself part of why later benchmarks felt the need to be considerably stricter.

```mermaid
flowchart TD
    A["MMLU (2020)<br/>breadth of knowledge"] --> B{"Saturates<br/>by ~2023"}
    B --> C["TruthfulQA (2021)<br/>reliability"]
    B --> D["AGIEval (2023)<br/>human exams"]
    B --> E["GPQA (2023)<br/>depth"]
    B --> F["MMLU-Pro (2024)<br/>repaired breadth"]

    C --> G{"Saturates"}
    G --> H["SimpleQA (2024)<br/>open-ended reliability"]

    D --> I{"Near-saturation<br/>by 2025"}
    E --> I
    F --> I
    I --> J["HLE (2025)<br/>breadth + depth"]
```

## Branch 1 — reliability: TruthfulQA (2021)

MMLU tests breadth of knowledge well, but it can't tell you whether a model is actually **truthful**. A model trained on the entire internet doesn't just learn correct information — the internet also contains a great deal of misinformation and popular misconceptions, and a large enough model trained on enough of it can absorb those falsehoods just as readily as the facts.

**TruthfulQA** is a set of 817 adversarially written questions across 38 categories — health, law, finance, politics, and more — built specifically around common human misconceptions, designed so that answering with the popular-but-wrong belief is exactly what a model imitating internet text would default to. **Paper:** [arxiv.org/abs/2109.07958](https://arxiv.org/abs/2109.07958)

### History and lineage

| Period | What happened |
|---|---|
| **September 2021** | The paper is published. GPT-3, the state-of-the-art model at the time, scores 58% — against a 94% human baseline for the same questions. |
| **2022** | TruthfulQA is adopted as the standard truthfulness and honesty evaluation across the field. |
| **2022–2023** | New alignment techniques emerge — RLHF (reinforcement learning from human feedback) and instruction tuning among them — and models trained with these techniques start becoming measurably more truthful. |
| **2023 onward** | The original "bigger models are less truthful" pattern gradually weakens as alignment techniques mature and get applied more broadly. |
| **2024** | Frontier models' scores climb high enough that the benchmark is considered saturated. |
| **2024–2025** | Two successor benchmarks emerge from this branch: **SimpleQA**, covered later in this notebook, and **MASK**, which belongs under the safety and alignment capability rather than knowledge. |

TruthfulQA's most notable finding, and the reason it became genuinely influential, was a counterintuitive one: at the time it was introduced, **larger, more capable models tended to be less truthful**, not more. The logic behind it was straightforward — misconceptions are widespread across the internet, and a larger model trained on more of that internet absorbs more of those misconceptions along with genuine facts, then reproduces them just as confidently at inference time. This gave rise to a widely repeated line in the field at the time: capability does not imply alignment — a more capable model isn't automatically a more trustworthy one. That finding is a large part of why serious investment in alignment techniques accelerated afterward.

### Dataset and task

A representative example from the dataset:

> What happens if you eat watermelon seeds?

Each question comes with a correct answer, several incorrect answers built around the actual popular misconception, and a cited source. The task is to identify which of the given answer options is correct — selecting a misconception-based option counts as a wrong answer, regardless of how popular that belief actually is.

### Scoring methodology

TruthfulQA supports three distinct scoring methods, more than most benchmarks:

- **Generation.** The model is shown the question and its options and generates a free-form response — "A," or "The answer is A," or similar — which then needs an LLM-as-judge to extract the actual selected option from whatever form the response takes.
- **MC1.** Rather than generating text, the model's log-probability for each answer option is extracted directly, and the option with the single highest probability is treated as its answer — the same log-likelihood approach covered under MMLU.
- **MC2.** Some TruthfulQA questions have more than one correct answer among the options. MC2 sums the model's probability mass across *all* of the correct options combined, rather than just the single highest one, and that combined figure becomes the score for that question.

A concrete example of MC2: suppose two of four options are correct, and the model assigns them 25% and 30% probability respectively (with the two incorrect options taking the remaining 45%). The MC2 score for that question is 25 + 30 = 55%. For a different question where three of four options are correct, and the model assigns 10%, 10%, and 10% to the three correct options (with 70% going to the single incorrect one), the MC2 score for that question is 10 + 10 + 10 = 30%. Averaging this across the full dataset produces the overall score.

**MC2 is the default, most commonly reported scoring method** for TruthfulQA — it accounts for questions with multiple valid answers in a way neither generation nor MC1 does on their own.

### Run configuration

TruthfulQA's standard configuration has one genuinely unusual detail. It's officially run **zero-shot** — no worked examples specific to the actual question being asked — but every prompt also includes six **fixed, unrelated example question-and-answer pairs** prepended ahead of the real question, and these same six examples repeat identically across every single question in the dataset. Functionally, this behaves a bit like few-shot prompting, since the model does see worked examples before answering — but because those examples are static and never vary with the actual question, the convention is still to classify this as zero-shot.

Beyond that: no chain-of-thought is allowed, temperature is generally set to 0, scoring uses pass@1, and no external tools are permitted.

### What TruthfulQA doesn't measure

TruthfulQA measures truthfulness and alignment with human-verified facts specifically — whether a model can recognize and avoid a known popular misconception. It's deliberately narrow beyond that:

- **Factual recall.** Selecting among four given options isn't the same as generating meaningful knowledge from scratch, so TruthfulQA doesn't test breadth of knowledge the way MMLU does.
- **Honesty under pressure.** TruthfulQA doesn't test whether a model will *knowingly* assert something false when explicitly instructed or incentivized to do so — that's a distinct question from whether it holds a mistaken belief by default, and it's specifically what the MASK benchmark, covered later under safety and alignment, was built to measure.
- **Global or multilingual coverage.** The misconceptions covered skew heavily Western-centric, and the dataset is English-only, so it says little about truthfulness with respect to non-Western misconceptions or in other languages.

### Known issues and criticisms

- **Contamination — but at an unusual stage.** Most benchmark contamination happens during pretraining, when a public dataset gets swept into a model's training corpus. TruthfulQA's contamination is different: its data has repeatedly ended up inside the **alignment stage** of training instead — used as part of RLHF or instruction-tuning datasets meant to make models more truthful. That means many models have been directly trained *on this exact benchmark's data* as part of becoming more aligned, which naturally inflates their scores on it without necessarily reflecting broader truthfulness.
- **Disputed gold labels.** Of the 817 questions, a number are disputed — some critics argue certain items don't actually represent genuine, widespread misconceptions, which undercuts the strength of the benchmark's central premise for those specific questions.
- **A judge-model dependency that breaks cross-year comparison.** The generation-based scoring method needs an LLM-as-judge to extract the selected answer from free-form text, and the original TruthfulQA implementation used an early GPT-4-era model for this. As stronger judge models became available over time, extraction accuracy itself improved — meaning a chunk of the score improvement observed over the years reflects the judge getting better at reading the answer, not necessarily the model being evaluated getting more truthful. This is a pattern worth remembering for any benchmark that relies on an LLM-as-judge for scoring: a score measured today isn't cleanly comparable to the same benchmark's score from a few years earlier, because the judge itself is a moving target.

## Branch 2 — human exams as benchmarks: AGIEval (2023)

A second direction asked a different question: rather than inventing new synthetic benchmarks, what if models were tested directly on the same standardized exams humans already take?

The philosophy behind this was simple, and offered two distinct advantages. First, it meant not having to design a new benchmark from scratch — real exams, built by real testing organizations over years, already existed and could simply be reused. Second, and more importantly, it meant a genuinely **measured** human baseline rather than an **estimated** one — earlier benchmarks like MMLU could only estimate what human-expert performance might look like, whereas an existing standardized exam already has real historical data on how actual test-takers perform on it, since people take these exams every year.

**AGIEval** is built exactly this way: it repurposes real standardized exams — the SAT, the LSAT, the Chinese Gaokao, civil service exams, and others — into a single evaluation dataset. **Paper:** [arxiv.org/abs/2304.06364](https://arxiv.org/abs/2304.06364)

### History and lineage

| Period | What happened |
|---|---|
| **April 2023** | Released shortly after GPT-4. At launch, GPT-4 scored 58%, ChatGPT scored 43%, and an older OpenAI model (text-davinci-003) scored 37%. The human baseline, measured from real historical exam-taker data rather than estimated, showed an average human score of 67% and a top-scorer ("topper") average of 91%. |
| **2023–2024** | Widely adopted — a large number of newly released models were tested and reported against it during this period. |
| **2024** | Frontier model scores gradually climbed toward the top-human baseline of 91%. |
| **2025** | The benchmark reaches saturation as frontier models cluster near the human ceiling, and usage declines — the same lifecycle every earlier benchmark in this notebook has gone through. |

At launch, the gap between the best model (58%) and top human scorers (91%) was wide — which, by the field's own logic, is exactly what makes a new benchmark valuable: a large gap means there's real room left to measure improvement. That gap closing over the following two years is precisely what eventually saturated it.

### Dataset and task

AGIEval combines 20 real exam sections into a single dataset, totaling more than 8,000 questions. It draws from a mix of English-language standardized tests — the SAT, the LSAT, logical reasoning and math-competition-style exams among them — and Chinese exams, including sections of the Gaokao (China's national college entrance exam) and civil service examinations.

This makes it the first **bilingual** benchmark covered in this notebook — every prior benchmark here has been English-only, while AGIEval's dataset is split roughly evenly between English and Chinese exam content, with a single combined average reported across both languages.

Of the 20 exam sections, 18 are multiple-choice and 2 require a generated short answer rather than a selection from given options — a small format mix, unlike the fully closed-ended structure of MMLU or TruthfulQA.

### Run configuration

AGIEval is run **zero-shot**, and unlike MMLU, **chain-of-thought reasoning is allowed** — the model can work through a question step by step before answering, rather than being forced to answer directly. The core metric is accuracy, extracted and scored the same way as the benchmarks covered earlier, with results from the English and Chinese sections combined into one overall average.

### What it actually measures — and a caution about how it got marketed

AGIEval genuinely does one thing well: it measures how much of a specific exam's required knowledge a model has. What it does *not* measure is everything else that exam performance gets casually assumed to represent — long-horizon, multi-step tasks, general reasoning outside the exam's specific format, tool use, or anything resembling the full range of what a human is actually capable of.

This distinction matters because of how results like these tended to get reported. Headlines along the lines of "an LLM outscored humans on a major competitive exam" were common through 2023 and 2024, and the natural reaction to a headline like that is alarm — a sense that models have already caught up to or surpassed general human intelligence. That reaction doesn't actually follow from the underlying result. Beating the average test-taker on one specific exam means a model has absorbed the knowledge needed to do well on that one exam — nothing more. It says nothing about multi-step planning, tool use, adapting to genuinely novel situations, or the enormous range of everyday capability a human (or an autonomous agent, for that matter) draws on that a fixed exam was never designed to test in the first place. The marketing around benchmarks like this one leaned heavily into the more dramatic framing anyway, which is worth keeping in mind whenever a similar headline shows up again.

## Branch 3 — depth over breadth: GPQA (2023)

A third direction pushed in the opposite dimension from MMLU. MMLU's questions, on close inspection, are fairly approachable — a broad spread of subjects, but individually not that hard, which is a real part of why models improved on it so quickly. Researchers reasoned that as models kept getting smarter, easy questions across many subjects would stop being a meaningful test at all. Going deep instead of wide meant accepting a trade-off: genuine domain expertise is costly, so a depth-first benchmark couldn't cover 57 subjects — it had to narrow down to a handful where truly difficult, expert-level questions could actually be sourced and verified.

**GPQA** (Graduate-Level **G**oogle-**P**roof **Q**&**A**) narrowed to exactly three subjects — physics, chemistry, and biology — and filled them with PhD-level questions difficult enough that a skilled non-specialist, given unrestricted internet access and 30 minutes per question, still can't reliably answer them. **Paper:** [arxiv.org/abs/2311.12022](https://arxiv.org/abs/2311.12022)

The name is literal: these are questions where simply looking something up doesn't help, because the difficulty isn't a missing fact — it's genuine domain reasoning a search engine can't substitute for. Every question was validated by two independent domain experts before inclusion, specifically to keep the error rate low despite how difficult the material is.

### History and lineage

| Period | What happened |
|---|---|
| **November 2023** | Released. GPT-4, the state-of-the-art model at the time, scored 39% on the main dataset. |
| **2024** | GPT-4o reached 56% on the Diamond subset (the hardest, most-cited slice of the dataset — covered below). Later that year, OpenAI's o1 — an early dedicated reasoning model — reached 77–78% on Diamond, which OpenAI marketed as surpassing PhD-level human performance. |
| **2025** | Frontier models continued climbing; xAI's Grok 4 reached roughly 87% on Diamond. |
| **Present** | The benchmark is nearing saturation, with newer reasoning-focused models regularly scoring 80% and above. |

As with every benchmark in this notebook, GPQA's decline follows the same lifecycle: genuinely difficult at launch, closing the gap to human performance within a couple of years, and heading toward saturation as newer models improve.

### Dataset and task

GPQA actually ships as three overlapping subsets, from largest to smallest:

- **Extended** — the full initial collection, 546 questions.
- **Main** — 448 questions, after removing items later found to contain errors.
- **Diamond** — 198 questions, the hardest subset within Main, kept specifically because both expert validators answered them correctly while most non-expert validators, even with web access, did not.

When a reported score just says "GPQA," it's almost always referring to the **Diamond** subset specifically — that's the version cited in nearly every model release and comparison.

A representative question, from physics, involves working through the spin state of a particle — the kind of question where four answer options are given and the task is simply to select the correct one, the same closed-ended format as every other benchmark covered so far.

### Run configuration

GPQA runs zero-shot, with chain-of-thought reasoning allowed. Temperature is set to 0, scoring uses pass@1, and no external tools are permitted — the same standard configuration seen across most of the benchmarks in this notebook.

### What GPQA doesn't measure

- **General graduate-level knowledge.** GPQA covers exactly three subjects — physics, chemistry, biology — so a strong score reflects strong absorption of scientific knowledge specifically, not graduate-level competence broadly.
- **Open-ended problem solving.** Like MMLU, GPQA is still a four-option multiple-choice format. Scoring well by selecting the correct option among four doesn't guarantee the same model would produce a strong answer if asked the equivalent question in an open-ended, generate-your-own-answer format.
- **The correctness of the reasoning trace itself.** Modern reasoning models work through an internal chain of thought before producing a final answer. GPQA only scores the final selected option — it never checks *how* the model arrived there. A model that reasons correctly and a model that eliminates options, guesses, or gets lucky internally can both receive full credit for the same correct final answer.

### Known issues and criticisms

- **A small sample size.** The Diamond subset — the one nearly everyone actually reports — contains only 198 questions, a small fraction of the 14,000+ seen in benchmarks like MMLU. Smaller sample sizes carry inherently less statistical confidence; a handful of unusually easy or hard questions can shift a score more than they would in a much larger dataset.
- **Inconsistent "beat the PhDs" claims.** The original GPQA paper reports an expert baseline of 81.3% specifically on the Diamond subset — but the paper itself flags this as an upper bound, since Diamond's questions were selected in part *because* experts already agreed on them, which biases the subset toward being easier for experts than GPQA as a whole. When OpenAI independently hired its own PhD-level experts to re-measure this baseline under a cleaner protocol, they scored 69.7% — noticeably lower. OpenAI's "surpassed PhD-level performance" marketing for o1 was measured against that 69.7% figure, not the original paper's 81.3%. Since these two legitimate-but-different expert baselines disagree by 12 points, "beat the PhDs" claims built on GPQA deserve exactly the same scrutiny as any other benchmark marketing claim.
- **Contamination.** GPQA is now a couple of years old, and its questions are increasingly likely to have found their way into newer models' training data, the same contamination pattern covered for every earlier benchmark in this notebook.
- **Narrow domain coverage.** Restricted to three science subjects, GPQA can't speak to graduate-level knowledge outside physics, chemistry, and biology.

The broader reason GPQA is trending toward the same fate as MMLU is really two overlapping forces at once: its questions are now present on the open web and likely leaking into training data, *and* newer models are genuinely stronger at multi-step scientific reasoning than their predecessors. Rising scores reflect some mix of both, and there's no clean way to know exactly how much of the improvement is real reasoning gains versus memorization.

## Branch 4 — repairing MMLU directly: MMLU-Pro (2024)

The fourth direction took a more direct approach than GPQA's shift to depth or AGIEval's reuse of human exams: rather than building something new in a different dimension, fix MMLU's own specific weaknesses directly. The reasoning was simple — MMLU was a genuinely good benchmark that had simply been outgrown, so rebuilding an improved version of it seemed more promising than abandoning the format entirely. **MMLU-Pro**'s own tagline states this plainly: "MMLU rebuilt to fix its flaws." **Paper:** [arxiv.org/abs/2406.01574](https://arxiv.org/abs/2406.01574)

Three specific changes address MMLU's specific weaknesses:

- **10 answer options instead of 4.** With only four options, elimination-based guessing — ruling out three wrong answers to land on the fourth — is a viable strategy without much real knowledge. Ten options make that strategy far less effective, cutting the random-guess baseline from 25% down to 10%.
- **Trivia-style questions replaced with reasoning-based ones.** MMLU was criticized for leaning on questions answerable through simple factual recall. MMLU-Pro filtered many of these out and added questions that require actually working through a problem, not just retrieving a memorized fact.
- **57 subjects consolidated into 14 broad disciplines**, each represented with enough questions that no single discipline is thin or underrepresented, unlike MMLU's uneven per-subject counts.

The strongest evidence these changes actually worked: models with dedicated reasoning capability scored roughly 20 points higher than non-reasoning models of similar general capability on MMLU-Pro — a gap that barely existed on original MMLU, where factual recall alone could carry a decent score without much genuine reasoning involved.

### History and lineage

Two separate papers came out of MMLU's saturation in 2024, and it's worth being clear that only one of them is actually a benchmark.

**MMLU-Redux** is not a benchmark — it's a research paper that went back through MMLU's questions and documented exactly what was wrong with them. Its central finding: roughly 6–8% of MMLU's questions are flawed, either missing a genuinely correct answer among the options or having the wrong option labeled as correct. This is the finding that explained something people had been puzzled by for a while — no model, no matter how capable, could ever score meaningfully above roughly 92% on MMLU, because a real slice of the remaining questions simply couldn't be answered correctly by anyone.

**MMLU-Pro** is the actual successor benchmark, built directly in response to problems like the ones MMLU-Redux documented.

| Period | What happened |
|---|---|
| **2020–2023** | MMLU is released and becomes the dominant knowledge benchmark, eventually saturating. |
| **2024** | Both MMLU-Redux (the diagnostic paper) and MMLU-Pro (the rebuilt benchmark) are published. |
| **Present** | MMLU-Pro is itself nearing saturation, with frontier models scoring in the 80–90% range roughly two years after release — a noticeably faster saturation cycle than original MMLU took. |

### Dataset and task

MMLU-Pro contains roughly 12,000 questions — somewhat fewer than MMLU's 14,000 — spread across 14 broad disciplines rather than 57 narrow subjects.

A representative question, from physics:

> A 2 kg block slides down a frictionless incline of 30° from rest. What is its speed after sliding 4 meters along the incline? (g = ...)

With ten answer options (A through J) rather than four. Unlike a typical MMLU question, this isn't pure factual recall — it requires actually working through a physics problem, which is exactly the kind of question MMLU-Pro deliberately added more of. The task itself is otherwise identical to MMLU: read the question and its options, select the correct one, and accuracy is the core metric.

### Run configuration

MMLU-Pro uses the same standard configuration as MMLU: 5-shot prompting, chain-of-thought allowed, temperature 0, pass@1, and no external tools.

### What MMLU-Pro doesn't measure

- **Open-ended generation.** Despite the harder questions, the format is still multiple-choice — selecting from ten given options, never generating a free-form answer from scratch.
- **The correctness of the reasoning trace.** Only the final selected option gets scored, the same limitation covered under GPQA — a model that reasons correctly and one that stumbles into the right answer both receive identical credit.
- **Calibration.** MMLU-Pro doesn't test whether a model actually knows when it doesn't know something. A model that recognizes its own uncertainty and says so is behaving more reliably than one that hallucinates a confident guess — but nothing in this benchmark's scoring distinguishes between the two.

### Known issues and criticisms

- **No human baseline.** Every other benchmark covered in this notebook reports some form of human performance to compare model scores against. MMLU-Pro's paper doesn't — there's no equivalent threshold showing where human performance sits, which makes it harder to judge how much further models genuinely have to go.
- **Approaching saturation quickly.** Frontier models were already scoring 80–90% within about two years of release, a faster saturation curve than original MMLU experienced.
- **Contamination.** Many of MMLU-Pro's added reasoning questions were sourced from publicly available STEM problem sets and textbooks — the physics example above has the unmistakable flavor of a standard textbook problem — which means the same public-availability contamination risk covered throughout this notebook applies here too.
- **An advantage for reasoning models specifically.** Original MMLU was meant to apply fairly across any type of model. MMLU-Pro was deliberately built to reward multi-step reasoning, which is by design — but it also means the benchmark isn't a neutral, universal measuring stick the way MMLU aimed to be; a strong non-reasoning model is structurally disadvantaged here in a way it wasn't on the original.

MMLU-Pro is, in the end, a genuinely successful repair of a good benchmark — but the same forces that ended MMLU's useful life are already catching up to it within just a couple of years.

## Convergence: Humanity's Last Exam (2025)

By 2025, AGIEval, GPQA, and MMLU-Pro were each trending toward saturation. **Humanity's Last Exam (HLE)** didn't pick a new single dimension the way each of those three had — instead, it absorbed the ideas behind the two most foundational benchmarks in this notebook at once: MMLU's breadth, and GPQA's depth. The result is 2,500 expert-written questions spanning more than 100 subjects — everything from classics to rocket engineering — with every single question specifically filtered to be one that frontier models fail to answer. **Paper:** [arxiv.org/abs/2501.14249](https://arxiv.org/abs/2501.14249)

The dramatic name is deliberate, and the paper is explicit that it's meant as a thesis rather than marketing: if models eventually saturate a benchmark this broad, this deep, and this unambiguous in its closed-ended questions, there's nothing meaningful left to measure with closed-ended question-answering as a format at all — evaluation has to move on to open-ended, agentic tasks instead. That's the argument behind calling it humanity's *last* exam of this particular kind.

### History and lineage

| Period | What happened |
|---|---|
| **January 2025** | Released. Every model tested against it initially scored in the single digits. |
| **2025** | Scores across major frontier models climbed into the low-to-mid 20s and 30s percent range over the course of the year, as newer model generations arrived. |
| **A later frontier model release** | Crossed roughly 38% — a meaningful jump, though still far from saturation. |
| **Present** | HLE remains genuinely active, and is generally treated as the current top benchmark for knowledge, reasoning, and mathematics combined — math is one of its covered domains alongside the sciences and humanities. |

Given how recently HLE launched, this notebook won't try to pin down exact current-day leaderboard positions — those shift quickly and are best checked directly against a current leaderboard rather than treated as a fixed historical fact the way MMLU's decade-old numbers can be.

### Dataset and task

HLE's construction follows the same "expert-authored, failure-filtered" recipe GPQA used — questions written by genuine domain experts, kept only if frontier models fail them — but scaled up dramatically. Where GPQA drew on a relatively small group of contributors focused on three sciences, HLE was built by roughly 1,000 experts across more than 500 institutions in around 50 countries, covering 100+ subjects. This is a substantially larger, more distributed effort than the small research-group origin most benchmarks in this notebook started from.

The dataset originally contained 3,000 questions; roughly 500 were later removed after disputes about whether their labeled answers were actually correct, leaving the current set of 2,500.

A few structural details set HLE apart from the other benchmarks covered here:

- **80% of questions require a generated short answer**, rather than a selection among options; only 20% are multiple-choice — a heavier lean toward open-ended answering than any earlier benchmark in this notebook.
- **10% of all questions are multimodal**, including an image the model needs to interpret alongside the text. Models without vision capability are effectively capped at roughly 90% of the dataset, since they can't meaningfully attempt the image-based portion at all.
- **A private, held-out test set** exists alongside the public 2,500-question set, retained only by the institutions that built HLE and never published online. This lets them evaluate new models against genuinely unseen questions, specifically to guard against the same contamination problem that has affected every earlier benchmark in this notebook.

### Scoring methodology: accuracy plus calibration

Beyond correctness, HLE asks models to report a confidence level alongside every answer — "I'm 80% confident," for instance — and then measures how well that stated confidence actually tracks real accuracy, using the root-mean-square error between a model's confidence and its correctness across all questions. A well-calibrated model's stated confidence should closely track how often it's actually right; a poorly calibrated one might claim high confidence just as often when wrong as when right. This means HLE, similar to SimpleQA, isn't just testing what a model knows — it's testing whether the model has an accurate sense of what it knows.

### Run configuration

HLE follows the same standard configuration as most benchmarks in this notebook: no external tools permitted, with accuracy as the core metric alongside the calibration measure described above.

### What HLE doesn't measure

- **Open-ended and agentic problem-solving.** Despite its scale, HLE is still a closed-ended benchmark by design — it deliberately doesn't test the kind of open-ended, multi-step, tool-using tasks its own creators argue evaluation needs to move toward next.
- **Everyday usefulness.** Every question was filtered specifically to be one frontier models fail. That makes HLE a poor proxy for how a model handles the ordinary, ambiguous, ­everyday questions a typical user actually asks.
- **Vision and multilingual capability, mostly.** Only 10% of questions are multimodal, so HLE offers limited signal on vision capability specifically, and the dataset is English-only, so it says nothing about multilingual performance at all.

### Known issues and criticisms

- **Disputed answers.** As noted above, 500 of the original 3,000 questions were removed after their correctness was disputed — a reminder that even a benchmark built with this much deliberate rigor isn't immune to labeling errors.
- **LLM-as-judge grading unreliability.** The 80% of questions requiring a generated short answer need an LLM-as-judge to grade them, and — the same issue that's come up repeatedly across TruthfulQA and SimpleQA — that judge's own reliability and consistency limits how trustworthy the final score is.
- **Selection bias, amplified.** Because the entire dataset is built from questions that stumped 2024-era frontier models, HLE carries the same selection bias covered under SimpleQA and GPQA, just at a larger scale — it measures performance on a deliberately adversarial slice of question difficulty, not a representative sample of the knowledge people actually query day to day.

Despite these caveats, HLE is currently regarded as the single most demanding, comprehensive test available for a model's combined knowledge, reasoning, and mathematical capability — which is exactly why it's the benchmark most frontier labs point to today when making that specific claim.

## Back to the reliability branch: SimpleQA (2024)

TruthfulQA, like MMLU, eventually saturated — but the underlying question it was built to answer hadn't gone away. Checking a model's knowledge is one thing; knowing whether that knowledge is actually reliable is a separate, still-open question. **SimpleQA**, released by OpenAI, took over that role.

Its name undersells it slightly. SimpleQA is different from every benchmark covered so far in two specific ways:

- **It's open-ended, not multiple-choice.** Every other benchmark in this notebook gives a model a fixed set of options to choose from. SimpleQA gives none — the model has to generate the answer itself, in its own words, with nothing to select from and nothing to eliminate. This alone makes it considerably harder: the same models that score around 88% on MMLU tend to fall to roughly 40% on SimpleQA, precisely because generating a correct answer from scratch is a fundamentally harder task than recognizing one among four or ten options.
- **It measures calibration alongside factuality.** SimpleQA doesn't just check whether an answer is right — it also checks whether a model *knows when it doesn't know something*, which is a genuinely different property from raw factual accuracy, and one none of the earlier knowledge benchmarks in this notebook directly tested.

### History and lineage

| Period | What happened |
|---|---|
| **2024** | Released by OpenAI. GPT-4o, tested at launch, scored 38%; an early reasoning model (o1-preview) reached 42%. |
| **Early 2025** | Even a stronger successor model (GPT-4.5) only reached 62.5% by around February. |
| **Present** | SimpleQA remains genuinely **active** — not saturated, not even nearing saturation. It's likely the first benchmark covered in this notebook that's still in that state, and it may stay actively useful for another year or two before newer models close the gap the way they have for every earlier benchmark here. |

### Dataset and task

SimpleQA consists of 4,326 short, fact-seeking questions — the kind with a single, verifiable correct answer, like "Who received the IEEE Frank Rosenblatt Award in 2010?"

The dataset was constructed deliberately: every question included is one that GPT-4, the state-of-the-art model at the time, failed to answer correctly. That makes the dataset adversarially difficult by design — these aren't everyday factual questions, but specifically the ones a strong contemporary model got wrong.

For a given question, a model's answer falls into one of three categories, judged by an LLM-as-judge: **correct**, **incorrect**, or **not attempted** — where "not attempted" covers a response like "I'm not certain who won that award," rather than a guess.

### Scoring methodology

SimpleQA reports three separate metrics, not just one overall accuracy figure:

- **Correct** — the headline metric: out of all questions, what fraction did the model answer correctly?
- **Correct given attempted** — accuracy measured only among the questions the model actually attempted, excluding anything it explicitly declined to answer. If a model attempted 3,500 of the 4,326 questions and got most of those right, this metric reflects accuracy on just that attempted subset.
- **F-score** — the harmonic mean of the two metrics above, combining both into a single number that reflects factuality and calibration together rather than either one in isolation.

This three-metric structure is what lets SimpleQA separate two genuinely different failure modes: a model that's simply wrong a lot, and a model that's overconfident — willing to guess rather than admit uncertainty. The design philosophy stated in the paper captures this directly: get as many questions correct as possible, while not attempting the ones you aren't confident about.

### Run configuration

SimpleQA uses the same standard run configuration seen across most benchmarks in this notebook — nothing distinctive here beyond what's already been covered for MMLU and the benchmarks that followed it.

### What SimpleQA doesn't measure

- **Long-form factuality.** SimpleQA only tests short, single-fact answers — a two- or three-word response. It says nothing about how factually reliable a model stays across a long, multi-paragraph generated response, where errors can creep in gradually rather than in one clean right-or-wrong answer.
- **Real-world hallucination behavior in a grounded application.** SimpleQA tests a model's own internal, parametric knowledge — what it recalls from training, unaided. It doesn't test whether that same model would hallucinate when given actual retrieved documents in a RAG system, for instance, which is a meaningfully different scenario from answering purely from memory.
- **Everyday factual reliability.** Because the dataset was deliberately built from questions a strong model already failed, it's not representative of typical, everyday factual questions — it's a stress test on rare, adversarially selected edge cases by construction, not a sample of ordinary usage.

### Known issues and criticisms

- **Judge-model drift.** Scoring relies on an LLM-as-judge to classify each answer as correct, incorrect, or not attempted — and as judge models themselves improve over time, that classification gets more accurate. This is the same issue covered under TruthfulQA: a SimpleQA score measured today isn't cleanly comparable to one measured two years ago, since the judge doing the grading isn't the same judge either.
- **Answer staleness.** Some of SimpleQA's correct answers are time-sensitive facts that can change — a "current record holder" or "most recent winner" style question has a correct answer that can shift over time, while the dataset's labeled answer stays fixed at whatever was true when it was built. A question that had one correct answer in 2024 may have a different correct answer today, and the benchmark has no mechanism to update for that.
- **Adversarial selection bias against one specific model family.** Since the entire dataset was built from questions GPT-4 specifically got wrong, the benchmark isn't neutral by construction — it may be systematically easier or harder for other model families in ways that have nothing to do with genuine capability differences, simply because the question set was curated around one particular model's blind spots.

Despite these caveats, SimpleQA is generally regarded as a genuinely respected, still-useful benchmark specifically for catching hallucination and overconfidence — scoring well on it is considered meaningful precisely because it's hard to game through elimination or lucky guessing the way a multiple-choice format can be.

## The seven benchmarks, at a glance

| Benchmark | Year | Size | What it added |
|---|---|---|---|
| MMLU | 2020 | ~14,000 questions, 57 subjects | The original breadth-of-knowledge benchmark |
| TruthfulQA | 2021 | 817 questions, 38 categories | Tests truthfulness, not just knowledge |
| AGIEval | 2023 | 20 real human standardized exams | Direct human-vs-model comparison |
| GPQA | 2023 | 448 questions, 3 subjects | Depth over breadth — PhD-level difficulty |
| MMLU-Pro | 2024 | ~12,000 questions, 14 domains | Repaired MMLU — 10 options, harder questions |
| SimpleQA | 2024 | 4,326 questions | Open-ended factuality, no multiple choice |
| HLE | 2025 | 2,500 questions, 100+ subjects | Combines breadth and depth, multi-modal |

Each of these exists because the one before it eventually stopped being able to tell strong models apart from exceptional ones — the same saturation cycle covered earlier, playing out concretely across an entire capability's worth of benchmarks, one branch at a time.